# Plotting keyword counts with seaborn

In Lecture 7 we searched for **energy** in UN General Debate speeches and saved one row for every mention. Now we turn those results into two charts.

Our questions:

1. Which countries mention energy most often?
2. Does the ranking change when we account for the length of their speeches?

We use the same 200 English-language speeches: 20 countries, 2015 to 2024. Each chart combines all ten years for each country.

Run cells with **Shift + Enter**. Cells with a `# ...` comment are for you to complete. The **Show solution** boxes provide help when needed.

## 1. Open our tools

`pandas` handles the tables. `seaborn` draws the charts. `matplotlib.pyplot` lets us set the chart size and display it. We reuse `re` for the same word cleaning as Lecture 7.

If an import fails, open a terminal in your course project folder and run `uv add pandas seaborn matplotlib`. Then restart the notebook kernel and run from the top.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

## 2. Load the keyword hits

Open this notebook from the `Lecture_8` folder. `..` means the parent folder, so `../Lecture_7/energy_hits.csv` points to last week's results.

In [ ]:
hits = pd.read_csv("../Lecture_7/energy_hits.csv")
hits.head()

<details>
<summary>FileNotFoundError?</summary>

Keep `Lecture_7` and `Lecture_8` next to each other, as in the course repository. If you downloaded only this notebook, use the course data online instead:

```python
base_url = "https://raw.githubusercontent.com/dsl-unibe-ch/wti_python_course/main/Lecture_7/"
hits = pd.read_csv(base_url + "energy_hits.csv")
```

Later, load the speeches with `speeches = pd.read_csv(base_url + "ungd_speeches.csv")`.

</details>

In [ ]:
len(hits)

<details>
<summary>Show solution</summary>

```python
len(hits)
```

There are 247 rows. Each row is one mention, rather than one speech.

</details>

## 3. Count mentions by country

We already used `value_counts()` last week. It counts how often each country appears in the hits table.

In [ ]:
hits["country_name"].value_counts()

<details>
<summary>Show solution</summary>

```python
hits["country_name"].value_counts()
```



</details>

For seaborn, we want an ordinary table with two named columns. `.rename("energy_count")` names the counts. `.reset_index()` turns the country names into a column again.

In [ ]:
country_counts = hits["country_name"].value_counts().rename("energy_count").reset_index()
country_counts.head()

## 4. Our first chart: total mentions

We put the counts on the horizontal axis (`x`) and the countries on the vertical axis (`y`). This leaves room to read the country names.

`order` controls the country order. Our table is already sorted from the highest count to the lowest. Each row contains one total, so `errorbar=None` turns off uncertainty bars.

**Predict:** which country will have the longest bar?

In [ ]:
plt.figure(figsize=(9, 7))
ax = sns.barplot(
    data=country_counts,
    x="energy_count",
    y="country_name",
    order=country_counts["country_name"],
    errorbar=None,
)
ax.set_title('Mentions of "energy" in UN speeches, 2015-2024')
ax.set_xlabel("Total mentions")
ax.set_ylabel("Country")
plt.tight_layout()
plt.show()

**Checkpoint:** Brazil has 31 mentions, followed by Saudi Arabia with 28 and Australia with 25.

These are counts in our selected speeches. They do not tell us how much energy a country produces or consumes.

**Discuss:** if one country gives much longer speeches, does it have more opportunities to mention energy?

## 5. Account for speech length

We need a denominator: the total number of words in each country's speeches. The hits table contains only the matches and cannot tell us the length of all the speeches, including those without a match.

We therefore return to the original speeches.

In [ ]:
speeches = pd.read_csv("../Lecture_7/ungd_speeches.csv")
speeches.head()

### Count the words in each speech

This is the cleaning recipe from Lecture 7: lower case, then find groups of letters a to z. We count exactly those tokens for the denominator too.

We collect one word count for every speech, keeping the original row order.

In [ ]:
word_counts = []
for text in speeches["text"]:
    words = re.findall(r"[a-z]+", text.lower())
    word_counts.append(len(words))

speeches["word_count"] = word_counts
speeches[["country_name", "year", "word_count"]].head()

### Add the words across all years

We use `groupby()` and `sum()`, as in the pandas lectures. One row will now represent one country across all ten years.

In [ ]:
country_words = speeches.groupby("country_name")["word_count"].sum().reset_index()
country_words.head()

<details>
<summary>Show solution</summary>

```python
country_words = speeches.groupby("country_name")["word_count"].sum().reset_index()
country_words.head()
```



</details>

### Bring the counts and lengths together

This is a merge from Pandas II. Starting with the word totals and using `how="left"` keeps every country in the speeches, even if it never mentions the keyword. A missing mention count becomes zero.

In [ ]:
country_summary = country_words.merge(country_counts, on="country_name", how="left")
country_summary["energy_count"] = country_summary["energy_count"].fillna(0).astype(int)
country_summary.head()

### Mentions per 1,000 words

The calculation is:

`mentions per 1,000 words = total mentions / total words * 1000`

For example, 10 mentions in 5,000 words gives 2 mentions per 1,000 words. Ten mentions in 10,000 words gives 1 mention per 1,000 words.

We divide the **total mentions by the total words** for each country. We are not taking the average of ten separate yearly rates.

In [ ]:
country_summary["energy_per_1000"] = (
    country_summary["energy_count"] / country_summary["word_count"] * 1000
)
country_summary.head()

<details>
<summary>Show solution</summary>

```python
country_summary["energy_per_1000"] = (
    country_summary["energy_count"] / country_summary["word_count"] * 1000
)
country_summary.head()
```



</details>

In [ ]:
country_summary = country_summary.sort_values("energy_per_1000", ascending=False)
country_summary

## 6. Our second chart: a rate and a new colour

We reuse the first chart with three changes: the value on `x`, the order of the countries, and `color="darkorange"`. The title and axis label must describe the new measure.

`sns.set_theme(style="whitegrid")` changes the background style of subsequent charts. The grid helps us read the values.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 7))
ax = sns.barplot(
    data=country_summary,
    x="energy_per_1000",
    y="country_name",
    order=country_summary["country_name"],
    color="darkorange",
    errorbar=None,
)
ax.set_title('Mentions of "energy" per 1,000 words, 2015-2024')
ax.set_xlabel("Mentions per 1,000 speech words")
ax.set_ylabel("Country")
plt.tight_layout()
plt.show()

### Your turn

Change the colour to `"steelblue"`, `"seagreen"`, or another matplotlib colour name, and run the chart cell again.

Then compare the two charts:

1. Which country ranks first in each chart?
2. Find a country that moves up after accounting for speech length.
3. Does changing the colour change the values or the ranking?
4. What question does each chart answer?

A rate makes texts of different lengths easier to compare. It still does not measure a country's policy priorities directly; the speeches, years, language and keyword all shape the result.

## 7. Save the table

Keep both the counts and the denominator so someone else can check the rate.

In [ ]:
country_summary.to_csv("energy_by_country.csv", index=False)

## What we did

| Task | Code |
|---|---|
| Count mentions by country | `hits["country_name"].value_counts()` |
| Make a plotting table | `.rename("energy_count").reset_index()` |
| Draw horizontal bars | `sns.barplot(data=..., x=..., y=...)` |
| Add up speech lengths | `.groupby("country_name")["word_count"].sum()` |
| Combine counts and lengths | `.merge(..., on="country_name", how="left")` |
| Calculate a comparable rate | `total_mentions / total_words * 1000` |
| Change the bar colour | `color="darkorange"` |

The speeches are from the United Nations General Debate Corpus, the same source used in Lecture 7.